In [ ]:
# Cell 1: config + browser startup.
import importlib
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "browser").exists():
    ROOT = ROOT.parent.resolve()
if not (ROOT / "browser").exists():
    raise RuntimeError("Could not locate repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

browser_driver = importlib.import_module("browser.driver")
core_config = importlib.import_module("core.config")
service_interact = importlib.import_module("services.web.interact")
service_markdown = importlib.import_module("services.web.markdown")
service_webagent = importlib.import_module("services.web.webagent")
llm_runtime = importlib.import_module("tests.llm_test.llm_runtime")

browser_driver = importlib.reload(browser_driver)
core_config = importlib.reload(core_config)
service_interact = importlib.reload(service_interact)
service_markdown = importlib.reload(service_markdown)
service_webagent = importlib.reload(service_webagent)
llm_runtime = importlib.reload(llm_runtime)

from browser.driver import build_driver
from core.config import load_app_config
from storage.embedded_mongo import EmbeddedMongoStore

app_config = load_app_config()
browser_cfg = dict(app_config["profile"]["browser"])
browser_cfg["headless"] = True
browser_cfg["user_data_dir"] = str(ROOT / "runtime" / "chrome_user_data")
openai_api_key_path = Path(r"D:\_Desktop\api_key.txt")
store = EmbeddedMongoStore(Path(app_config["profile"]["mongo_file"]))
driver = build_driver(browser_cfg)
LLM_TEST_DIR = ROOT / "tests" / "llm_test"
markdown_text = ""
interactables = {
    "interactables": [],
    "images": [],
    "rows": [],
    "counts": {"interactables": 0, "images": 0, "rows": 0},
}
session_outputs = []
runtime_state = {
    "driver": driver,
    "store": store,
    "log_path": "mock://mongodb/webagent",
    "api_key_path": str(openai_api_key_path),
    "markdown_text": markdown_text,
    "interactables": interactables,
    "session_outputs": session_outputs,
    "phase": "planning",
    "last_plan": "",
    "last_tool": "",
    "last_tool_result": None,
    "delays": {
        "webagent_click": 2,
        "webagent_type": 2,
        "webagent_clear_text": 2
    },
    "context_tokens": 20000,
    "response_reserve_tokens": 2048,
    "verbose": True,
    "wait_seconds": 0,
    "current_web_state": {},
}

print(json.dumps({"root": str(ROOT), "browser": browser_cfg}, indent=2))


In [ ]:
# Cell 2: runtime loop.
from typing import Any

service_interact = importlib.import_module("services.web.interact")
service_markdown = importlib.import_module("services.web.markdown")
service_webagent = importlib.import_module("services.web.webagent")
llm_runtime = importlib.import_module("tests.llm_test.llm_runtime")
service_interact = importlib.reload(service_interact)
service_markdown = importlib.reload(service_markdown)
service_webagent = importlib.reload(service_webagent)
llm_runtime = importlib.reload(llm_runtime)

from tests.llm_test.llm_runtime import (
    append_transcript_text,
    build_messages,
    call_model_chat_with_retry,
    display_block,
    display_json,
    display_markdown_block,
    execute_tool_call,
    format_tool_result_for_display,
    format_tool_result_for_llm,
    _tool_target_text,
    load_instruction_bundle,
    trim_messages_to_budget,
    parse_model_output,
    stop_requested,
)

INSTRUCTIONS = load_instruction_bundle(LLM_TEST_DIR)
MODEL_BACKEND = "local"
OPENAI_MODEL = "gpt-5.4-mini"
OPENAI_API_BASE = "https://api.openai.com/v1/chat/completions"
LLAMA_MODEL = "qwen3.5-9b"
LLAMA_API_BASE = "http://127.0.0.1:8080/v1/chat/completions"


def run_agent(task: str, max_steps: int = 20) -> dict[str, Any]:
    runtime = runtime_state
    runtime["task"] = task
    session_outputs.clear()
    runtime["session_outputs"] = session_outputs
    runtime["stuck_counts"] = {}
    runtime["phase"] = "planning"
    runtime["last_plan"] = ""
    runtime["last_tool"] = ""
    runtime["last_tool_result"] = None
    runtime["current_web_state"] = {}
    runtime["memory"] = {
        "task": task,
        "phase": "planning",
        "plan": "",
        "last_tool": "",
        "last_result": "",
        "findings": [],
        "errors": [],
        "current_web_state": {},
        "session_state": {},
    }

    display_block("User task", task, kind="user")
    context_tokens = int(runtime.get("context_tokens", 20000))
    response_reserve_tokens = int(runtime.get("response_reserve_tokens", 2048))

    for step in range(max_steps):
        if stop_requested():
            final_text = "Loop halted by ] key."
            display_markdown_block("Final response", final_text, kind="final")
            runtime["last_result"] = {"kind": "halted", "text": final_text}
            return runtime["last_result"]

        messages = build_messages(task, INSTRUCTIONS, phase=runtime["phase"], runtime=runtime)
        messages = trim_messages_to_budget(
            messages,
            max_context_tokens=context_tokens,
            response_reserve_tokens=response_reserve_tokens,
            keep_head=len(messages),
        )

        try:
            llm_text = call_model_chat_with_retry(
                MODEL_BACKEND,
                messages=messages,
                api_key_path=runtime["api_key_path"],
                openai_model=OPENAI_MODEL,
                openai_base_url=OPENAI_API_BASE,
                openai_reasoning_effort="low",
                llama_model=LLAMA_MODEL,
                llama_base_url=LLAMA_API_BASE,
                max_completion_tokens=1024,
                timeout=300,
                retries=2,
            )
        except Exception as exc:
            error_text = str(exc)
            display_block(f"LLM error {step + 1}", error_text, kind="error")
            runtime["last_result"] = {"kind": "error", "message": error_text}
            return runtime["last_result"]
        display_block(f"LLM response {step + 1}", llm_text, kind="llm")
        append_transcript_text(runtime, "llm", f"step-{step + 1}", llm_text)

        parsed = parse_model_output(llm_text)
        if parsed["kind"] == "final_response":
            display_markdown_block("Final response", parsed["text"], kind="final")
            runtime["last_result"] = parsed
            append_transcript_text(runtime, "final", "response", parsed["text"])
            return parsed

        if parsed["kind"] == "error":
            display_json("Parsed error", parsed, kind="error")
            messages.append(
                {
                    "role": "user",
                    "content": "The previous response was invalid. Return exactly one <cmd>...</cmd> or one <final_response>...</final_response>.",
                }
            )
            messages = trim_messages_to_budget(
                messages,
                max_context_tokens=context_tokens,
                response_reserve_tokens=response_reserve_tokens,
                keep_head=len(messages),
            )
            key = parsed.get("message", "parse-error")
            runtime["stuck_counts"][key] = runtime["stuck_counts"].get(key, 0) + 1
            if runtime["stuck_counts"][key] > 4:
                final_text = f"Stopped after repeated parse errors: {key}"
                display_markdown_block("Final response", final_text, kind="final")
                runtime["last_result"] = {"kind": "stopped", "text": final_text}
                return runtime["last_result"]
            continue

        if parsed["kind"] == "text":
            if runtime["phase"] == "planning":
                display_block(f"Plan {step + 1}", parsed["text"], kind="plan")
                runtime["last_plan"] = parsed["text"]
                append_transcript_text(runtime, "plan", f"step-{step + 1}", parsed["text"])
                runtime["phase"] = "tool"
                continue
            if runtime["phase"] == "reflection":
                display_block(f"Reflection {step + 1}", parsed["text"], kind="plan")
                append_transcript_text(runtime, "reflection", f"step-{step + 1}", parsed["text"])
                lowered = parsed["text"].strip().casefold()
                if lowered.startswith("decision: complete"):
                    display_markdown_block("Final response", parsed["text"], kind="final")
                    runtime["last_result"] = {"kind": "final_response", "text": parsed["text"]}
                    return runtime["last_result"]
                runtime["phase"] = "tool"
                continue
            display_block(f"Unexpected text {step + 1}", parsed["text"], kind="error")
            runtime["phase"] = "tool"
            messages.append(
                {
                    "role": "user",
                    "content": "The previous response was invalid. Return exactly one <cmd>...</cmd> or one <final_response>...</final_response>. Do not use plain English for a tool step.",
                }
            )
            continue

        if parsed["kind"] == "command":
            cmd_text = f'<cmd>{parsed["raw"]}</cmd>'
            display_block(f"Parsed cmd {step + 1}", cmd_text, kind="cmd")
            result = execute_tool_call(parsed["tool"], parsed["args"], runtime)
            output_text = format_tool_result_for_llm(result, runtime)
            target_text = _tool_target_text(result)
            output_label = f"Tool output {step + 1}"
            if target_text:
                output_label += f" - {target_text}"
            display_block(output_label, format_tool_result_for_display(result, runtime), kind="output")
            append_transcript_text(runtime, "cmd", f"step-{step + 1}", cmd_text)
            append_transcript_text(runtime, "output", f"step-{step + 1}", result)

            stuck_key = f'{parsed["tool"]}:{result.get("status", "unknown")}:{result.get("message", "")}'
            runtime["stuck_counts"][stuck_key] = runtime["stuck_counts"].get(stuck_key, 0) + 1
            if runtime["stuck_counts"][stuck_key] > 4:
                final_text = f'Stopped after repeating the same tool state: {stuck_key}'
                display_markdown_block("Final response", final_text, kind="final")
                runtime["last_result"] = {"kind": "stopped", "text": final_text, "last_tool": parsed["tool"]}
                return runtime["last_result"]

            runtime["last_tool"] = parsed["tool"]
            runtime["last_tool_result"] = result
            runtime["phase"] = "reflection"
            continue

        display_json("Unexpected parse result", parsed, kind="error")
        runtime["phase"] = "tool"

    final_text = f"Stopped after max_steps={max_steps}."
    display_markdown_block("Final response", final_text, kind="final")
    runtime["last_result"] = {"kind": "stopped", "text": final_text}
    return runtime["last_result"]


# Example call.
# run_agent("Open the page and inspect it with webuse tools.")


In [ ]:
# Cell 3: run a task example.
# Edit the task string below, then run this cell.

runtime_state["task"] = ""
runtime_state["phase"] = "planning"
runtime_state["last_plan"] = ""
runtime_state["last_tool"] = ""
runtime_state["last_tool_result"] = None
runtime_state["current_web_state"] = {}
runtime_state["memory"] = {
    "task": "",
    "phase": "planning",
    "plan": "",
    "last_tool": "",
    "last_result": "",
    "findings": [],
    "errors": [],
    "current_web_state": {},
    "session_state": {},
    "last_reflection": "",
    "next_action": "",
}
session_outputs.clear()
runtime_state["session_outputs"] = session_outputs
driver.get("about:blank")
runtime_state["current_web_state"] = {}
runtime_state["markdown_text"] = ""
runtime_state["interactables"] = {
    "interactables": [],
    "images": [],
    "rows": [],
    "counts": {"interactables": 0, "images": 0, "rows": 0},
}

task = "Go to DuckDuckGo and search for the cheapest VPS server available."
result = run_agent(task)
result
